In [29]:
import sys
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values


In [30]:
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

SOURCE_DIR = PROJECT_ROOT / "data" / "bronze" / "open_meteo" / "archive"

print("Archive source directory:", SOURCE_DIR)
print("UTC now:", datetime.now(timezone.utc).isoformat())

Archive source directory: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive
UTC now: 2026-08-07T14:28:28.183191+00:00


In [31]:
def connect():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DATABASE"),
        user=os.getenv("POSTGRES_USERNAME"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )


def get_archive_files(directory: Path):
    return sorted(directory.rglob("*.json"))


def build_archive_rows(path: Path) -> list[tuple]:
    path = Path(path)
    raw = pd.read_json(path)
    hourly = raw.get("hourly", {})

    if isinstance(hourly, pd.Series):
        hourly = hourly.to_dict()

    times = pd.Series(hourly.get("time", [])).tolist()
    if not times:
        return []

    field_mapping = {
        "temperature_2m": "temperature_celsius",
        "apparent_temperature": "feels_like_celsius",
        "relative_humidity_2m": "relative_humidity_pct",
        "dew_point_2m": "dew_point_2m",
        "precipitation": "precipitation_mm",
        "rain": "rain",
        "snowfall": "snowfall",
        "snow_depth": "snow_depth",
        "weather_code": "weather_code",
        "wind_speed_10m": "wind_speed_10m",
        "wind_direction_10m": "wind_direction_10m",
    }

    data = {
        "source_name": "open_meteo",
        "source_type": "archive",
        "city_name": path.stem.rsplit("_", 1)[0],
        "ingestion_date": path.parent.name,
        "observation_time": [f"{observation_time}Z" for observation_time in times],
    }

    for source_field, target_field in field_mapping.items():
        data[target_field] = pd.Series(hourly.get(source_field, [])).tolist()

    df = pd.DataFrame(data)
    return [tuple(row) for row in df.itertuples(index=False, name=None)]

In [32]:
archive_files = get_archive_files(SOURCE_DIR)

all_rows = []
skipped = []

for path in archive_files:
    rows = build_archive_rows(path)
    if rows:
        all_rows.extend(rows)
    else:
        skipped.append(str(path))

print(f"Found {len(archive_files)} archive files")
print(f"Built {len(all_rows)} rows")
if skipped:
    print(f"Skipped {len(skipped)} invalid files")

Found 68 archive files
Built 3360 rows


In [33]:
# Removido "precipitation_probability" da query SQL
insert_sql = """
INSERT INTO silver.archive_weather_hourly (
    source_name, 
    source_type, 
    city_name, 
    ingestion_date, 
    observation_time,
    temperature_celsius, 
    feels_like_celsius, 
    relative_humidity_pct,
    dew_point_2m,
    precipitation_mm, 
    rain,
    snowfall,
    snow_depth,
    weather_code,
    wind_speed_10m, 
    wind_direction_10m
) VALUES %s;
"""

if all_rows:
    with connect() as conn:
        with conn.cursor() as cursor:
            execute_values(cursor, insert_sql, all_rows)
        conn.commit()
    print("Loaded rows into silver.archive_weather_hourly")
else:
    print("Nenhuma linha para inserir.")

Loaded rows into silver.archive_weather_hourly
